# Step 5a — Analysis of the pre-annotation of the equal-data-set (schema v0.2)

**Purpose.** Describe the result of the pre-annotation made in Step 5: the size of the annotated corpus and the distribution of flagged spans by class.

**Input.** `outputs/s5_preannotation/preannotation_spans.csv`, `preannotation_units.csv` and `ad_context.csv` (all from Step 5; masked text only).

**Definitions used in the tables**
- **Instance** = one unit (a job title, or one line/bullet/sentence of qualification or body text) from Steps 1–3.
- **Source advertisement** = one of the 1,500 rows of `equal-data-set.xlsx`.
- **Token** = a word or punctuation mark in the masked unit text, counted with the regular expression `\w+|[^\w\s]` (a simple word-level count, not sub-word pieces of any model).
- **Flagged span** = one span the lexicon/rule pre-annotation marked, whatever its label, including spans labelled not-bias or left for review.
- **Class** = the v0.2 label code. **Group** = the bias dimension (all labels here belong to the Gender schema; labels outside it are shown as *Not bias / review*). **Type** = Explicit, Implicit, Not bias (inclusive listing, polisemantic) or Unsettled (the rules could not decide).

**What this is not.** Counts describe what the rules flagged. They are not a measure of how much gender bias the dataset contains, because no human has confirmed any label; I1/I2 in particular are single lexicon words marked for review.

In [1]:
import re
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 220)
P = Path("outputs") / "s5_preannotation"
OUT = Path("outputs") / "s5a_analysis"
OUT.mkdir(parents=True, exist_ok=True)

spans = pd.read_csv(P / "preannotation_spans.csv", encoding="utf-8-sig")
units = pd.read_csv(P / "preannotation_units.csv", encoding="utf-8-sig")
ads = pd.read_csv(P / "ad_context.csv", encoding="utf-8-sig")
TOKEN = re.compile(r"\w+|[^\w\s]")
units["tokens"] = units["text_masked"].map(lambda t: len(TOKEN.findall(str(t))))
print(len(units), "units;", len(spans), "span rows;", len(ads), "ads")

7971 units; 1218 span rows; 1500 ads


## 1. Corpus size

In [2]:
flag_units = units["n_candidate_spans"] > 0
spans_tok = spans["span_text"].map(lambda t: len(TOKEN.findall(str(t))))
corpus = pd.DataFrame([
    ("Instances (sentences / requirement units)", f"{len(units):,}"),
    ("Source advertisements", f"{units['record_id'].nunique():,}"),
    ("Tokens", f"{int(units['tokens'].sum()):,}"),
    ("Flagged spans", f"{len(spans):,}"),
], columns=["Measure", "Value"])
display(corpus)

extra = pd.DataFrame([
    ("Instances with at least one flagged span", f"{int(flag_units.sum()):,} ({flag_units.mean():.1%})"),
    ("Advertisements with at least one flagged span", f"{int((ads['n_candidate_spans'] > 0).sum()):,} ({(ads['n_candidate_spans'] > 0).mean():.1%})"),
    ("Tokens inside flagged spans", f"{int(spans_tok.sum()):,} ({spans_tok.sum() / units['tokens'].sum():.2%} of all tokens)"),
    ("Tokens per instance: mean / median / max", f"{units['tokens'].mean():.1f} / {units['tokens'].median():.0f} / {units['tokens'].max()}"),
    ("Instances per advertisement: mean / median / max", f"{ads['n_units'].mean():.1f} / {ads['n_units'].median():.0f} / {ads['n_units'].max()}"),
    ("Flagged spans per flagged instance: mean / max", f"{len(spans) / flag_units.sum():.2f} / {units['n_candidate_spans'].max()}"),
], columns=["Additional measure", "Value"])
display(extra)

by_comp = units.groupby("component").agg(instances=("unit_id", "size"), ads=("record_id", "nunique"), tokens=("tokens", "sum"),
                                          instances_flagged=("n_candidate_spans", lambda s: int((s > 0).sum()))).reset_index()
by_comp["flagged_spans"] = by_comp["component"].map(spans.groupby("component").size()).fillna(0).astype(int)
display(by_comp)

,Measure,Value
0,Instances (sentences / requirement units),"7,971"
1,Source advertisements,"1,500"
2,Tokens,"58,064"
3,Flagged spans,"1,218"


,Additional measure,Value
0,Instances with at least one flagged span,"1,074 (13.5%)"
1,Advertisements with at least one flagged span,765 (51.0%)
2,Tokens inside flagged spans,"1,457 (2.51% of all tokens)"
3,Tokens per instance: mean / median / max,7.3 / 6 / 59
4,Instances per advertisement: mean / median / max,5.3 / 5 / 20
5,Flagged spans per flagged instance: mean / max,1.13 / 5


,component,instances,ads,tokens,instances_flagged,flagged_spans
0,body_other,100,23,779,16,17
1,job_title,1497,1497,3818,90,90
2,qualification,6374,1343,53467,968,1111


## 2. Class distribution

Class names follow `schema_v0.2.json`. Rows for codes the rules left unsettled (`needs_review`, `E4_or_E5`) and the one rule-rejected row are shown as their own lines so that the total equals the number of flagged spans.

In [3]:
NAMES = {
    "E1": "Gender Restriction / Preference", "E2": "Explicit Marking of Sex", "E3": "Gendered Neologism",
    "E4": "Generic He", "E5": "Generic She", "I1": "Masculine Coded", "I2": "Feminine Coded",
    "INC": "Inclusive (both genders listed)", "P": "Polisemantic", "O": "Others",
    "E4_or_E5": "Generic he / she (undecided)", "needs_review": "Gender word, unclassified", "none": "Rejected by rule (no gender word)",
}
TYPE = {"E1": "Explicit", "E2": "Explicit", "E3": "Explicit", "E4": "Explicit", "E5": "Explicit", "I1": "Implicit", "I2": "Implicit",
        "INC": "Not bias", "P": "Not bias", "O": "Not bias", "E4_or_E5": "Unsettled", "needs_review": "Unsettled", "none": "Rejected"}
GROUP = {k: ("Gender" if TYPE[k] in ("Explicit", "Implicit") else "Not bias / review") for k in TYPE}
ORDER = list(NAMES)

d = spans.copy()
d["Class"] = d["v02_candidate"].map(lambda c: f"{c} {NAMES[c]}")
d["Group"] = d["v02_candidate"].map(GROUP)
d["Type"] = d["v02_candidate"].map(TYPE)
dist = (d.groupby(["v02_candidate", "Class", "Group", "Type"])
          .agg(Spans=("unit_id", "size"), Instances=("unit_id", "nunique"), Ads=("record_id", "nunique"),
               Distinct_wordings=("span_text", lambda s: s.str.lower().nunique())).reset_index())
dist["_o"] = dist["v02_candidate"].map(ORDER.index)
dist = dist.sort_values("_o").drop(columns=["_o", "v02_candidate"]).reset_index(drop=True)
dist["Share_of_spans"] = (dist["Spans"] / dist["Spans"].sum()).map("{:.1%}".format)
total = pd.DataFrame([{"Class": "Total", "Group": "", "Type": "", "Spans": dist["Spans"].sum(), "Instances": int(flag_units.sum()),
                       "Ads": int((ads["n_candidate_spans"] > 0).sum()), "Distinct_wordings": spans["span_text"].str.lower().nunique(), "Share_of_spans": "100.0%"}])
class_table = pd.concat([dist, total], ignore_index=True)
display(class_table)

,Class,Group,Type,Spans,Instances,Ads,Distinct_wordings,Share_of_spans
0,E1 Gender Restriction / Preference,Gender,Explicit,317,312,311,11,26.0%
1,E2 Explicit Marking of Sex,Gender,Explicit,40,40,32,6,3.3%
2,I1 Masculine Coded,Gender,Implicit,350,324,240,41,28.7%
3,I2 Feminine Coded,Gender,Implicit,379,324,275,40,31.1%
4,INC Inclusive (both genders listed),Not bias / review,Not bias,99,99,98,5,8.1%
5,P Polisemantic,Not bias / review,Not bias,11,11,11,1,0.9%
6,E4_or_E5 Generic he / she (undecided),Not bias / review,Unsettled,5,3,2,3,0.4%
7,"needs_review Gender word, unclassified",Not bias / review,Unsettled,16,16,16,3,1.3%
8,none Rejected by rule (no gender word),Not bias / review,Rejected,1,1,1,1,0.1%
9,Total,,,1218,1074,765,108,100.0%


### By Type and Group

In [4]:
type_table = (d.groupby(["Group", "Type"]).agg(Spans=("unit_id", "size"), Instances=("unit_id", "nunique"), Ads=("record_id", "nunique")).reset_index()
                .sort_values("Spans", ascending=False))
type_table["Share_of_spans"] = (type_table["Spans"] / len(d)).map("{:.1%}".format)
display(type_table)

bias = d[d["Type"].isin(["Explicit", "Implicit"])]
adlvl = pd.DataFrame([
    ("Ads with an explicit span (E1-E5)", ads["record_id"].isin(d.loc[d["Type"] == "Explicit", "record_id"]).sum()),
    ("Ads with an implicit span (I1-I2)", ads["record_id"].isin(d.loc[d["Type"] == "Implicit", "record_id"]).sum()),
    ("Ads with both", len(set(d.loc[d["Type"] == "Explicit", "record_id"]) & set(d.loc[d["Type"] == "Implicit", "record_id"]))),
    ("Ads with an implicit span only", len(set(d.loc[d["Type"] == "Implicit", "record_id"]) - set(d.loc[d["Type"] == "Explicit", "record_id"]))),
    ("Ads with an INC listing only (no E/I span)", len(set(d.loc[d["Type"] == "Not bias", "record_id"]) - set(bias["record_id"]))),
], columns=["Advertisement-level view (rule candidates)", "Ads"])
adlvl["Share_of_1500_ads"] = (adlvl["Ads"] / len(ads)).map("{:.1%}".format)
display(adlvl)

,Group,Type,Spans,Instances,Ads,Share_of_spans
1,Gender,Implicit,729,595,413,59.9%
0,Gender,Explicit,357,352,339,29.3%
2,Not bias / review,Not bias,110,110,108,9.0%
4,Not bias / review,Unsettled,21,19,18,1.7%
3,Not bias / review,Rejected,1,1,1,0.1%


,Advertisement-level view (rule candidates),Ads,Share_of_1500_ads
0,Ads with an explicit span (E1-E5),339,22.6%
1,Ads with an implicit span (I1-I2),413,27.5%
2,Ads with both,68,4.5%
3,Ads with an implicit span only,345,23.0%
4,Ads with an INC listing only (no E/I span),72,4.8%


### Most frequent wordings per class

The same wording recurs across ads, so the dominant wordings show how much of a class is a few repeated strings.

In [5]:
w = (d.assign(wording=d["span_text"].str.lower())
       .groupby(["v02_candidate", "wording"]).size().rename("Spans").reset_index())
w["Class"] = w["v02_candidate"].map(lambda c: f"{c} {NAMES[c]}")
w["_o"] = w["v02_candidate"].map(ORDER.index)
w = w.sort_values(["_o", "Spans"], ascending=[True, False])
w["share_of_class"] = w["Spans"] / w.groupby("v02_candidate")["Spans"].transform("sum")
top = w.groupby("v02_candidate").head(6)[["Class", "wording", "Spans", "share_of_class"]].reset_index(drop=True)
top["share_of_class"] = top["share_of_class"].map("{:.1%}".format)
display(top)

conc = w.groupby("v02_candidate").apply(lambda g: pd.Series({"distinct_wordings": len(g), "top3_share": g["Spans"].head(3).sum() / g["Spans"].sum()}), include_groups=False)
conc["top3_share"] = conc["top3_share"].map("{:.1%}".format)
display(conc.loc[[c for c in ORDER if c in conc.index]])

,Class,wording,Spans,share_of_class
0,E1 Gender Restriction / Preference,male,192,60.6%
1,E1 Gender Restriction / Preference,female,100,31.5%
2,E1 Gender Restriction / Preference,preferably male,7,2.2%
3,E1 Gender Restriction / Preference,preferably female,6,1.9%
4,E1 Gender Restriction / Preference,male only,3,0.9%
5,E1 Gender Restriction / Preference,for female,2,0.6%
6,E2 Explicit Marking of Sex,foreman,22,55.0%
7,E2 Explicit Marking of Sex,salesman,13,32.5%
8,E2 Explicit Marking of Sex,man,2,5.0%
9,E2 Explicit Marking of Sex,boy,1,2.5%


,distinct_wordings,top3_share
v02_candidate,,
E1,11.0,94.3%
E2,6.0,92.5%
I1,41.0,41.7%
I2,40.0,51.7%
INC,5.0,98.0%
P,1.0,100.0%
E4_or_E5,3.0,100.0%
needs_review,3.0,100.0%
none,1.0,100.0%


## 3. Validation and export

In [6]:
checks = []
def rec(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": detail})

rec("class table spans sum to the number of flagged spans", int(dist["Spans"].sum()) == len(spans), f"{len(spans):,}")
rec("type table spans sum to the number of flagged spans", int(type_table["Spans"].sum()) == len(spans))
rec("instances equal rows of the unit file and are unique", units["unit_id"].is_unique and len(units) == 7971, f"{len(units):,}")
rec("source advertisements equal the 1,500 ads of the data set", units["record_id"].nunique() == 1500 == len(ads))
rec("component tokens sum to the corpus token total", int(by_comp["tokens"].sum()) == int(units["tokens"].sum()))
rec("component flagged spans sum to the total", int(by_comp["flagged_spans"].sum()) == len(spans))
rec("every class code has a name, group and type", set(spans["v02_candidate"]) <= set(NAMES))
rec("flagged instances equal units with a candidate in the unit file", int(flag_units.sum()) == spans["unit_id"].nunique())
validation = pd.DataFrame(checks)
display(validation)

corpus.to_csv(OUT / "corpus_summary.csv", index=False, encoding="utf-8-sig")
extra.to_csv(OUT / "corpus_additional_measures.csv", index=False, encoding="utf-8-sig")
by_comp.to_csv(OUT / "corpus_by_component.csv", index=False, encoding="utf-8-sig")
class_table.to_csv(OUT / "class_distribution.csv", index=False, encoding="utf-8-sig")
type_table.to_csv(OUT / "type_group_distribution.csv", index=False, encoding="utf-8-sig")
adlvl.to_csv(OUT / "ad_level_view.csv", index=False, encoding="utf-8-sig")
top.to_csv(OUT / "top_wordings_per_class.csv", index=False, encoding="utf-8-sig")
validation.to_csv(OUT / "validation_checks.csv", index=False, encoding="utf-8-sig")
assert validation["passed"].all(), validation[~validation["passed"]]
print("Saved:", *sorted(str(p) for p in OUT.iterdir()), sep="\n  ")

,check,passed,detail
0,class table spans sum to the number of flagged spans,True,"1,218"
1,type table spans sum to the number of flagged spans,True,
2,instances equal rows of the unit file and are unique,True,"7,971"
3,"source advertisements equal the 1,500 ads of the data set",True,
4,component tokens sum to the corpus token total,True,
5,component flagged spans sum to the total,True,
6,"every class code has a name, group and type",True,
7,flagged instances equal units with a candidate in the unit file,True,


Saved:
  outputs\s5a_analysis\ad_level_view.csv
  outputs\s5a_analysis\class_distribution.csv
  outputs\s5a_analysis\corpus_additional_measures.csv
  outputs\s5a_analysis\corpus_by_component.csv
  outputs\s5a_analysis\corpus_summary.csv
  outputs\s5a_analysis\top_wordings_per_class.csv
  outputs\s5a_analysis\type_group_distribution.csv
  outputs\s5a_analysis\validation_checks.csv
